# Synthetic Micromagnetic Landscape

This notebook uses lightweight synthetic LEM/NEB-like data to demonstrate the package data model and interactive PlotlyJS visualizations. The numbers are illustrative: they mimic minima, saddle energies, magnetic moments, and local magnetization textures without running a micromagnetic solver.

In [ ]:
using Pkg
example_dir = basename(pwd()) == "examples" ? pwd() : joinpath(pwd(), "examples")
if !isfile(joinpath(example_dir, "Project.toml"))
    example_dir = pwd()
end
Pkg.activate(example_dir)
Pkg.develop(path=abspath(joinpath(example_dir, "..")))
Pkg.instantiate()

using DisconnectivityGraphs
using LinearAlgebra
using PlotlyJS
using Random

## Synthetic Minima and Saddles

Each minimum stores an energy and a small amount of metadata. In a Merrill.jl adapter, this metadata would include grain id, mesh path, temperature, net magnetic moment, minimizer diagnostics, and a pointer to a saved node-magnetization state. Saddles represent NEB transition states connecting pairs of minima.

In [ ]:
moments = Dict(
    :plus_x => [1.0, 0.05, 0.00],
    :tilted_plus => [0.75, 0.45, 0.10],
    :vortex_up => [0.05, 0.05, 0.55],
    :vortex_down => [-0.05, 0.02, -0.55],
    :tilted_minus => [-0.75, -0.35, -0.08],
    :minus_x => [-1.0, -0.03, 0.00],
)

minima = [
    Minimum(:plus_x, 0.00; metadata=Dict(:moment => moments[:plus_x], :texture => :uniform_plus)),
    Minimum(:tilted_plus, 0.18; metadata=Dict(:moment => moments[:tilted_plus], :texture => :flower_plus)),
    Minimum(:vortex_up, 0.42; metadata=Dict(:moment => moments[:vortex_up], :texture => :vortex_up)),
    Minimum(:vortex_down, 0.48; metadata=Dict(:moment => moments[:vortex_down], :texture => :vortex_down)),
    Minimum(:tilted_minus, 0.25; metadata=Dict(:moment => moments[:tilted_minus], :texture => :flower_minus)),
    Minimum(:minus_x, 0.05; metadata=Dict(:moment => moments[:minus_x], :texture => :uniform_minus)),
]

saddles = [
    Saddle(:plus_x, :tilted_plus, 1.35; metadata=Dict(:path => "NEB plus to tilted plus")),
    Saddle(:tilted_plus, :vortex_up, 1.85),
    Saddle(:vortex_up, :vortex_down, 2.70),
    Saddle(:vortex_down, :tilted_minus, 1.95),
    Saddle(:tilted_minus, :minus_x, 1.30),
    Saddle(:tilted_plus, :tilted_minus, 3.25),
]

landscape = LandscapeGraph(minima, saddles)
tree = disconnectivity_tree(landscape)
leaf_order(tree)

## Disconnectivity Tree

The branch height records the saddle energy where two previously disconnected basins first become connected. This view is useful for seeing whether the transition matrix is controlled by one dominant bottleneck or by many comparable pathways.

In [ ]:
function plot_disconnectivity_tree(landscape; title="Synthetic disconnectivity tree")
    tree = disconnectivity_tree(landscape)
    layout = tree_layout(tree)
    xs = Float64[]
    ys = Float64[]
    for segment in layout.segments
        append!(xs, [segment.x0, segment.x1, NaN])
        append!(ys, [Float64(segment.y0), Float64(segment.y1), NaN])
    end

    minimum_by_id = Dict(m.id => m for m in landscape.minima)
    leaf_ids = leaf_order(tree)
    leaf_x = [layout.leaf_positions[id] for id in leaf_ids]
    leaf_y = [minimum_by_id[id].energy for id in leaf_ids]
    polarity = [dot(minimum_by_id[id].metadata[:moment], [1.0, 0.0, 0.0]) for id in leaf_ids]
    hover = ["$(id)<br>E=$(round(minimum_by_id[id].energy; digits=3))<br>m=$(round.(minimum_by_id[id].metadata[:moment]; digits=2))" for id in leaf_ids]

    traces = [
        scatter(x=xs, y=ys, mode="lines", line=attr(color="#334155", width=2.5), hoverinfo="skip", name="branches"),
        scatter(x=leaf_x, y=leaf_y, mode="markers+text", text=String.(leaf_ids), textposition="bottom center",
            marker=attr(size=12, color=polarity, colorscale="RdBu", cmin=-1, cmax=1,
                colorbar=attr(title="m · x̂"), line=attr(color="white", width=1)),
            hovertext=hover, hoverinfo="text", name="minima"),
    ]
    plot(traces, Layout(title=title, xaxis=attr(visible=false), yaxis=attr(title="relative energy"),
        width=850, height=520, plot_bgcolor="white", paper_bgcolor="white"))
end

plot_disconnectivity_tree(landscape)

## Synthetic 3D Magnetization States

This panel imitates the visual role of `plot_magnetization!` in Merrill.jl. The dropdown switches among synthetic LEM configurations while preserving the same grain geometry and camera.

In [ ]:
grid = range(-1.0, 1.0; length=5)
points = [[x, y, z] for x in grid for y in grid for z in grid if maximum(abs.([x, y, z])) <= 1.0]

normalize3(v) = norm(v) == 0 ? [1.0, 0.0, 0.0] : v ./ norm(v)

function synthetic_vectors(texture, points)
    vectors = Vector{Vector{Float64}}()
    for p in points
        x, y, z = p
        r2 = x^2 + y^2
        v = if texture == :uniform_plus
            [1.0, 0.04*y, 0.04*z]
        elseif texture == :uniform_minus
            [-1.0, 0.04*y, -0.04*z]
        elseif texture == :flower_plus
            [1.0, 0.22*y, 0.18*z]
        elseif texture == :flower_minus
            [-1.0, -0.18*y, -0.12*z]
        elseif texture == :vortex_up
            [-y, x, 0.55 * exp(-3r2)]
        elseif texture == :vortex_down
            [y, -x, -0.55 * exp(-3r2)]
        else
            [1.0, 0.0, 0.0]
        end
        push!(vectors, normalize3(v))
    end
    vectors
end

function cone_trace_for_minimum(minimum; visible=false)
    texture = minimum.metadata[:texture]
    vecs = synthetic_vectors(texture, points)
    cone(x=[p[1] for p in points], y=[p[2] for p in points], z=[p[3] for p in points],
         u=[v[1] for v in vecs], v=[v[2] for v in vecs], w=[v[3] for v in vecs],
         sizemode="absolute", sizeref=0.23, anchor="tail", colorscale="Viridis",
         showscale=false, visible=visible, name=String(minimum.id))
end

function plot_synthetic_state_dropdown(minima)
    traces = [cone_trace_for_minimum(m; visible=i == 1) for (i, m) in pairs(minima)]
    buttons = [attr(label=String(m.id), method="update",
        args=[attr(visible=[j == i for j in eachindex(minima)]),
              attr(title="Synthetic magnetization: $(m.id)")]) for (i, m) in pairs(minima)]
    layout = Layout(title="Synthetic magnetization: $(minima[1].id)", width=760, height=620,
        scene=attr(aspectmode="data", xaxis=attr(visible=false), yaxis=attr(visible=false), zaxis=attr(visible=false)),
        updatemenus=[attr(type="dropdown", x=0.02, y=1.08, buttons=buttons)])
    plot(traces, layout)
end

plot_synthetic_state_dropdown(minima)